# 02 — Semantic Retrieval

Learn how to find the source passage most relevant to an LLM claim.

## 1. Why Semantic Retrieval?

We need to find the source text most relevant to each claim. Embeddings help us compare text by meaning instead of exact words.

## Note on the Original Experiments

The original notebook contained experimental code. The Day 2 version below reorganizes the same learning goal into a structured sequence so that each experiment has an explanation and an observation.

Keep the original notebook as a backup if you want to compare your experimentation process.

## 2. Imports

Import the libraries needed for embeddings and similarity.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

## 3. Load the Embedding Model

Load a pretrained model that converts sentences into embeddings.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

## 4. Understanding Embeddings

Convert a sentence into a numerical vector.

In [ ]:
sentence = "The telescope was launched in 2021."

embedding = model.encode(sentence)

print("Embedding:")
print(embedding)
print("\nNumber of dimensions:", len(embedding))

### Observation

The sentence is represented as a 384-dimensional vector.

## 5. Measuring Semantic Similarity

Compare two sentence embeddings using cosine similarity.

In [ ]:
sentence1 = "The telescope was launched in 2021."
sentence2 = "The telescope began its mission in 2021."

embedding1 = model.encode(sentence1)
embedding2 = model.encode(sentence2)

similarity = cosine_similarity(
    [embedding1],
    [embedding2]
)[0][0]

print("Cosine similarity:", similarity)

## 6. Compare with an Unrelated Sentence

To understand the similarity score better, we compare the telescope sentence with an unrelated sentence.

In [ ]:
sentence3 = "The capital of France is Paris."

embedding3 = model.encode(sentence3)

unrelated_similarity = cosine_similarity(
    [embedding1],
    [embedding3]
)[0][0]

print("Similarity with unrelated sentence:", unrelated_similarity)

### Observation

Similarity helps us find relevant text. It does not tell us whether a claim is true.

## 7. Document Chunking

Split the source document into smaller chunks for retrieval.

In [ ]:
chunks = [
    "The James Webb Space Telescope is a space telescope designed to conduct infrared astronomy.",

    "It was launched on December 25, 2021, aboard an Ariane 5 rocket from the Guiana Space Centre.",

    "The telescope operates near the second Lagrange point, approximately 1.5 million kilometers from Earth."
]

print("Number of chunks:", len(chunks))

## 8. Embed Source Chunks

Convert every source chunk into an embedding.

In [ ]:
chunk_embeddings = model.encode(chunks)

print("Embedding matrix shape:", chunk_embeddings.shape)

### Observation

Three chunks produce an embedding matrix with shape `(3, 384)`.

## 9. Retrieve Evidence for a Claim

Find the source chunk most similar to the claim.

In [ ]:
claim = "The telescope was launched in 2021."

claim_embedding = model.encode(claim)

similarities = cosine_similarity(
    [claim_embedding],
    chunk_embeddings
)[0]

for i, score in enumerate(similarities):
    print(f"Chunk {i}: {score:.4f}")

## 10. Select the Most Relevant Chunk

Choose the chunk with the highest similarity score.

In [ ]:
best_index = similarities.argmax()

print("Best chunk index:", best_index)
print("Evidence:", chunks[best_index])
print("Similarity:", similarities[best_index])

## 11. Create a Reusable Retrieval Function

Combine the retrieval steps into one function.

In [ ]:
def retrieve_evidence(claim, chunks):
    claim_embedding = model.encode(claim)
    chunk_embeddings = model.encode(chunks)

    similarities = cosine_similarity(
        [claim_embedding],
        chunk_embeddings
    )[0]

    best_index = similarities.argmax()

    return {
        "claim": claim,
        "evidence": chunks[best_index],
        "similarity": similarities[best_index]
    }

## 12. Test the Retrieval Function

Test the function with one claim.

In [ ]:
result = retrieve_evidence(
    "The telescope was launched in 2021.",
    chunks
)

print("Claim:", result["claim"])
print("Evidence:", result["evidence"])
print("Similarity:", result["similarity"])

## 13. Test Multiple Claims

Test retrieval on several claims, including an intentionally incorrect claim.

In [ ]:
claims = [
    "The telescope was launched in 2021.",
    "The telescope was launched using an Ariane 5 rocket.",
    "The telescope is approximately 1.5 million kilometers from Earth.",
    "The telescope was launched from India."
]

for claim in claims:
    result = retrieve_evidence(claim, chunks)

    print("\nCLAIM:")
    print(result["claim"])

    print("\nEVIDENCE:")
    print(result["evidence"])

    print("\nSIMILARITY:")
    print(f"{result['similarity']:.4f}")

    print("-" * 60)

## 14. Retrieval Experiment

Check whether the correct evidence was retrieved for each claim.

## 15. Observations

Retrieval finds relevant evidence using semantic similarity, but it does not determine whether the claim is true.

## 16. Limitations

- Chunks are manually created.
- We test only a small document.
- We retrieve only the top-1 chunk.
- Similarity is not the same as factual correctness.

## 17. Next Step

Next, use an NLI model to decide whether the retrieved evidence supports, contradicts, or does not establish the claim.